# IEEE39 EMT SSN: zero-skipping and linear-solver comparison

This notebook runs the IEEE39 electromagnetic-transient case using only SSN three-phase EMT network components: SSN fourth-order synchronous generators, impedance loads, Pi lines, and transformers. It compares zero skipping on/off and dense/KLU linear solvers over five simulated seconds at a 50 µs timestep. It checks power-flow convergence, logged three-phase voltages, solver agreement, and stability.

The runner is `dpsim/examples/cxx/Grids/IEEE39_Ph3.cpp`. Build `IEEE39_Ph3` with KLU enabled, then set `DPSIM_IEEE39_BINARY` if the executable is outside the repository's standard build paths.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
REPO = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "dpsim" / "examples" / "cxx" / "Grids" / "IEEE39_Ph3.cpp").is_file()
    ),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from within a DPsim checkout.")

configured_binary = os.environ.get("DPSIM_IEEE39_BINARY")
binary_candidates = (
    [Path(configured_binary).expanduser()] if configured_binary else []
) + [
    REPO / "build-zero-skip" / "dpsim" / "examples" / "cxx" / "IEEE39_Ph3",
    REPO / "build" / "dpsim" / "examples" / "cxx" / "IEEE39_Ph3",
]
BINARY = next((p.resolve() for p in binary_candidates if p.is_file()), None)
if BINARY is None:
    raise FileNotFoundError("Build target IEEE39_Ph3 or set DPSIM_IEEE39_BINARY.")
print(f"Repository: {REPO}")
print(f"Executable: {BINARY}")
RUN_ROOT = (
    REPO
    / "output"
    / "ieee39-emt-zero-skipping-comparison"
    / time.strftime("%Y%m%d-%H%M%S")
)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run outputs: {RUN_ROOT}")

## Run all combinations

Each setting runs once by default for five seconds of EMT simulation. The runner uses a 50 µs step and logs every tenth step. Run folders and logs are saved under `output/ieee39-emt-zero-skipping-comparison/` in the repository checkout.

In [ ]:
REPEATS = 1  # Increase this for lower-noise timing averages.
EMT_FINAL_TIME = 5.0
EMT_TIME_STEP = 50e-6


def run_variant(skip, backend, repeat):
    run_dir = RUN_ROOT / f"{skip}-{backend}-run{repeat}"
    run_dir.mkdir(parents=True, exist_ok=True)
    command = [str(BINARY), "emt", skip, backend, str(EMT_FINAL_TIME)]
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=run_dir,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    elapsed = time.perf_counter() - started
    if completed.returncode != 0:
        raise RuntimeError(
            f"{command} failed (exit {completed.returncode}):\n{completed.stdout[-6000:]}"
        )
    sim_name = f"IEEE39_EMT3Ph_CPP_zero_skipping_{skip}_{backend}"
    csv_path = run_dir / "logs" / sim_name / f"{sim_name}.csv"
    if not csv_path.is_file():
        csvs = list(run_dir.rglob("*.csv"))
        raise FileNotFoundError(
            f"Expected EMT log not found. CSV files: {csvs}; output: {completed.stdout[-3000:]}"
        )
    data = pd.read_csv(csv_path, skipinitialspace=True)
    data.columns = data.columns.str.strip()
    pf_log_path = run_dir / "logs" / "IEEE39_SP_PF_PF_Solver.log"
    if not pf_log_path.is_file():
        raise FileNotFoundError(f"Power-flow solver log not found: {pf_log_path}")
    return {
        "skip": skip,
        "backend": backend,
        "seconds": elapsed,
        "data": data,
        "log": csv_path,
        "pf_log": pf_log_path.read_text(),
        "stdout": completed.stdout,
    }


runs = {}
for skip in ("on", "off"):
    for backend in ("dense", "klu"):
        samples = [run_variant(skip, backend, repeat) for repeat in range(REPEATS)]
        for result in samples:
            convergence = re.search(r"Converged in (\d+) iterations", result["pf_log"])
            assert convergence, f"Power flow did not converge for {skip}, {backend}."
            result["pf_iterations"] = int(convergence.group(1))
        runs[(skip, backend)] = {
            **samples[-1],
            "seconds": float(np.mean([result["seconds"] for result in samples])),
            "seconds_std": (
                float(np.std([result["seconds"] for result in samples], ddof=1))
                if len(samples) > 1
                else 0.0
            ),
        }
        print(
            f"zero skipping {skip:>3}, {backend:>5}: "
            f"{runs[(skip, backend)]['seconds']:.3f} ± {runs[(skip, backend)]['seconds_std']:.3f} s "
            f"(mean ± SD, {REPEATS} runs), PF converged in "
            f"{runs[(skip, backend)]['pf_iterations']} Newton iterations"
        )

## Numerical agreement and stability checks

All four EMT settings must produce finite, time-aligned voltage data through the requested final time. The notebook compares zero skipping on/off for each backend and KLU/dense for each skip setting. Stability is checked from the peak-voltage envelope in consecutive 0.1-second windows.

In [ ]:
reference = runs[("off", "dense")]["data"]
voltage_columns = [column for column in reference.columns if column != "time"]
assert voltage_columns

for key, result in runs.items():
    data = result["data"]
    assert voltage_columns == [column for column in data.columns if column != "time"]
    values = data[voltage_columns].to_numpy()
    assert np.isfinite(values).all(), f"Non-finite EMT voltage in variant {key}."
    np.testing.assert_allclose(data["time"], reference["time"], rtol=0, atol=1e-12)
    assert (
        data["time"].iloc[-1] >= EMT_FINAL_TIME - 10 * EMT_TIME_STEP
    ), f"Variant {key} stopped at t={data['time'].iloc[-1]} s"
    assert (
        float(np.max(np.abs(values))) < 2 * 345e3
    ), f"Voltage envelope outside expected bounds for {key}."

comparisons = {}
for dimension, pairs in {
    "zero skipping (dense)": (("on", "dense"), ("off", "dense")),
    "zero skipping (KLU)": (("on", "klu"), ("off", "klu")),
    "linear solver (skip on)": (("on", "klu"), ("on", "dense")),
    "linear solver (skip off)": (("off", "klu"), ("off", "dense")),
}.items():
    lhs = runs[pairs[0]]["data"][voltage_columns].to_numpy()
    rhs = runs[pairs[1]]["data"][voltage_columns].to_numpy()
    max_abs = float(np.max(np.abs(lhs - rhs)))
    relative = max_abs / max(float(np.max(np.abs(rhs))), 1.0)
    assert relative < 1e-4, f"{dimension} changed EMT voltage by {relative:g} relative."
    comparisons[dimension] = {
        "max abs difference (V)": max_abs,
        "max relative difference": relative,
    }

# Compare peak voltage in consecutive 0.1-second windows.
times = reference["time"].to_numpy()
window_duration = 0.1
num_windows = int(EMT_FINAL_TIME / window_duration)
stability = {}
for key, result in runs.items():
    values = result["data"][voltage_columns].to_numpy()
    window_peaks = []
    for window_index in range(num_windows):
        start = window_index * window_duration
        mask = (times >= start) & (times < start + window_duration)
        assert mask.any(), f"No logged samples in stability window {window_index}."
        window_peaks.append(float(np.max(np.abs(values[mask]))))
    envelope_spread = (max(window_peaks) - min(window_peaks)) / max(window_peaks)
    assert (
        envelope_spread < 0.05
    ), f"EMT voltage envelope is not stable across windows for {key}: {envelope_spread:.2%}"
    tail = times >= times[-1] - window_duration
    prior = (times >= times[-1] - 2 * window_duration) & (
        times < times[-1] - window_duration
    )
    drift = float(
        np.max(np.abs(values[tail].mean(axis=0) - values[prior].mean(axis=0)))
    )
    stability[key] = {
        "last-window mean drift (V)": drift,
        "0.1-second peak envelope spread": envelope_spread,
    }

summary = pd.DataFrame(
    [
        {
            "zero skipping": skip,
            "linear solver": backend,
            "wall time mean (s)": runs[(skip, backend)]["seconds"],
            "wall time SD (s)": runs[(skip, backend)]["seconds_std"],
            "zero-skipping speedup (x)": runs[("off", backend)]["seconds"]
            / runs[("on", backend)]["seconds"],
            "KLU speedup vs dense (x)": runs[(skip, "dense")]["seconds"]
            / runs[(skip, "klu")]["seconds"],
            "PF Newton iterations": runs[(skip, backend)]["pf_iterations"],
            **stability[(skip, backend)],
        }
        for skip in ("on", "off")
        for backend in ("dense", "klu")
    ]
)
display(summary)
display(pd.DataFrame(comparisons).T)
print(
    "Checks passed: four EMT configurations converged, stayed finite and bounded, "
    "and agreed within 1e-4 relative voltage."
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
plot_columns = [
    column
    for column in voltage_columns
    if any(column.startswith(f"N{bus}.V_") for bus in [1, 10, 20, 30, 31, 39])
]
for column in plot_columns:
    axes[0].plot(
        reference["time"],
        runs[("on", "klu")]["data"][column],
        linewidth=0.8,
        label=column,
    )
axes[0].set_ylabel("Voltage (V)")
axes[0].set_title("IEEE39 EMT voltage traces (zero skipping on, KLU, 50 µs step)")
axes[0].grid(alpha=0.25)
axes[0].legend(ncol=3, fontsize=8)

for title, pairs in {
    "Skip on/off, dense": (("on", "dense"), ("off", "dense")),
    "Skip on/off, KLU": (("on", "klu"), ("off", "klu")),
    "KLU/dense, skip on": (("on", "klu"), ("on", "dense")),
}.items():
    lhs = runs[pairs[0]]["data"][voltage_columns].to_numpy()
    rhs = runs[pairs[1]]["data"][voltage_columns].to_numpy()
    axes[1].semilogy(
        reference["time"],
        np.maximum(np.max(np.abs(lhs - rhs), axis=1), 1e-16),
        label=title,
    )
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Max voltage difference (V)")
axes[1].set_title("Maximum EMT trajectory difference across logged phases")
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()

## SSN validation run (2026-11-13)

Executed one five-second EMT run per configuration at a 50 µs step. Every EMT network element was an SSN three-phase component (fourth-order synchronous generators, impedance loads, Pi lines, and transformers). The separate SP power-flow initialization remains the same. Run logs and the executed notebook are saved under `output/ieee39-emt-zero-skipping-comparison/` in the repository checkout.

All four power flows converged in four Newton iterations. Voltage samples were finite and bounded; the 0.1-second peak-envelope spread was 0.000130–0.000138, and the last-window mean drift was about 1.40 kV. All solver/zero-skipping comparisons passed the notebook's 1e-4 relative voltage agreement check. Maximum differences: dense zero-skipping on/off 0 V; KLU on/off 5.677 V (1.92e-5 relative); dense/KLU with skipping on 5.475 V (1.85e-5 relative); dense/KLU with skipping off 0.345 V (1.13e-6 relative).

| Zero skipping | Solver | Wall time (s) | KLU speedup vs dense |
|:--|:--|--:|--:|
| On | Dense | 21.100 | 4.04× |
| On | KLU | 5.229 | — |
| Off | Dense | 20.963 | 3.51× |
| Off | KLU | 5.970 | — |

Zero skipping made dense about 0.7% slower and KLU about 12.4% faster in this single run. These timings are indicative, not a repeated performance benchmark.
